# Global Satisfaction Model — Trained on All Studies

This notebook trains a **single Ridge regression model** on the combined data from all studies,
then evaluates it:
- **Globally** (all studies pooled)
- **Per study** (Study 1, Study 4, Study 5 individually)

Each study's scores are **min-max normalized to a common [0, 1] scale** before training,
so that scale differences (e.g. Study 4 uses 1–10 while Study 5 could top out lower)
don't bias the global model. Predictions are reported both in normalized space and
back-transformed to the original scale of each study.

## 0. Imports & Setup

In [22]:
import pandas as pd
import numpy as np
import torch
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from sentence_transformers import SentenceTransformer
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 0

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [23]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    cache_folder='../../final_pipeline/models/embedding_models'
).to(device)
print("Embedding model loaded.")

Embedding model loaded.


## 1. Helper Functions

In [24]:
def safe_detect(text):
    if not isinstance(text, str):
        return "unknown"
    if len(text) < 15:
        return "en"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

def clean_data(df, review_col):
    df[review_col] = df[review_col].fillna("").astype(str)
    df['ID'] = df.index.astype(int) + 1
    df['lang'] = df[review_col].apply(safe_detect)
    return df

def compute_metrics(y_true, y_pred, label=""):
    """Compute MAE (raw & rounded) globally and for negative/positive splits."""
    y_pred_rounded = np.round(y_pred * 2) / 2

    mae        = mean_absolute_error(y_true, y_pred)
    mae_r      = mean_absolute_error(y_true, y_pred_rounded)

    mask_neg = y_true <= 4
    mask_pos = y_true > 4

    results = {
        "label":           label,
        "n_total":         len(y_true),
        "n_negative":      mask_neg.sum(),
        "n_positive":      mask_pos.sum(),
        "mae":             mae,
        "mae_rounded":     mae_r,
    }

    if mask_neg.sum() > 0:
        results["mae_neg"]         = mean_absolute_error(y_true[mask_neg], y_pred[mask_neg])
        results["mae_rounded_neg"] = mean_absolute_error(y_true[mask_neg], y_pred_rounded[mask_neg])
    else:
        results["mae_neg"] = results["mae_rounded_neg"] = np.nan

    if mask_pos.sum() > 0:
        results["mae_pos"]         = mean_absolute_error(y_true[mask_pos], y_pred[mask_pos])
        results["mae_rounded_pos"] = mean_absolute_error(y_true[mask_pos], y_pred_rounded[mask_pos])
    else:
        results["mae_pos"] = results["mae_rounded_pos"] = np.nan

    return results

def print_metrics(r):
    print(f"\n{'='*55}")
    print(f"  {r['label']}  (n={r['n_total']})")
    print(f"{'='*55}")
    print(f"  MAE (raw)              : {r['mae']:.4f}")
    print(f"  MAE (rounded ×0.5)     : {r['mae_rounded']:.4f}")
    print(f"  MAE negative (y≤4)     : {r['mae_neg']:.4f}  [n={r['n_negative']}]")
    print(f"  MAE rounded  negative  : {r['mae_rounded_neg']:.4f}")
    print(f"  MAE positive (y>4)     : {r['mae_pos']:.4f}  [n={r['n_positive']}]")
    print(f"  MAE rounded  positive  : {r['mae_rounded_pos']:.4f}")

## 2. Load & Prepare Each Study

In [26]:
# ── Study metadata ────────────────────────────────────────────────────────────
# Each entry: (study_label, file_path, review_col, satisfaction_col)
STUDIES = [
    ("Study 1", '../../data/initial_data/Study 1 reviews.xlsx', 'finalReview',  'Satisfaction_final'),
    ("Study 3", '../../data/initial_data/Study 3 reviews.xlsx', 'Review',       'Emotionality_1to9'),
    ("Study 4", '../../data/initial_data/Study 4 reviews.xlsx', 'text',          'Satisfaction_RA2'),
    ("Study 5", '../../data/initial_data/Study 5 reviews.xlsx', 'Review',        'Satisfaction_final'),
]

study_data = {}   # key → {"df": ..., "review_col": ..., "sat_col": ...}

for label, path, review_col, sat_col in STUDIES:
    df = pd.read_excel(path)
    df = clean_data(df, review_col)
    df = df.dropna(subset=[sat_col])
    study_data[label] = {"df": df, "review_col": review_col, "sat_col": sat_col}
    print(f"{label}: {len(df)} rows | score range [{df[sat_col].min()}, {df[sat_col].max()}]")

Study 1: 2602 rows | score range [1.0, 9.0]
Study 3: 392 rows | score range [1, 9]
Study 4: 623 rows | score range [1, 9]
Study 5: 717 rows | score range [1, 9]


## 3. Encode Reviews with Sentence Embeddings

In [27]:
for label, info in study_data.items():
    df = info["df"]
    review_col = info["review_col"]
    print(f"Encoding {label} ({len(df)} reviews)...")
    embeddings = embedding_model.encode(
        df[review_col].fillna("").tolist(),
        convert_to_tensor=False,
        show_progress_bar=True
    )
    study_data[label]["X"] = embeddings
    study_data[label]["y"] = df[info["sat_col"]].values.astype(float)
    print(f"  ✓ Embeddings shape: {embeddings.shape}")

Encoding Study 1 (2602 reviews)...


Batches:   0%|          | 0/82 [00:00<?, ?it/s]

Batches: 100%|██████████| 82/82 [00:03<00:00, 22.85it/s]


  ✓ Embeddings shape: (2602, 768)
Encoding Study 3 (392 reviews)...


Batches: 100%|██████████| 13/13 [00:01<00:00, 10.40it/s]


  ✓ Embeddings shape: (392, 768)
Encoding Study 4 (623 reviews)...


Batches: 100%|██████████| 20/20 [00:00<00:00, 74.35it/s]


  ✓ Embeddings shape: (623, 768)
Encoding Study 5 (717 reviews)...


Batches: 100%|██████████| 23/23 [00:00<00:00, 37.58it/s]

  ✓ Embeddings shape: (717, 768)


## 4. Build the Combined Dataset

We stack all embeddings and satisfaction scores into one array.
A `study_id` array keeps track of which study each row comes from,
enabling per-study evaluation later.

In [28]:
all_X      = []
all_y      = []
all_study  = []

for label, info in study_data.items():
    all_X.append(info["X"])
    all_y.append(info["y"])
    all_study.extend([label] * len(info["y"]))

X_all     = np.vstack(all_X)
y_all     = np.concatenate(all_y)
study_ids = np.array(all_study)

print(f"Combined dataset: {X_all.shape[0]} samples, {X_all.shape[1]} features")
print(f"Score range: [{y_all.min()}, {y_all.max()}]")
print("\nSamples per study:")
for label in study_data:
    print(f"  {label}: {(study_ids == label).sum()}")

Combined dataset: 4334 samples, 768 features
Score range: [1.0, 9.0]

Samples per study:
  Study 1: 2602
  Study 3: 392
  Study 4: 623
  Study 5: 717


## 5. Train the Global Model

In [29]:
clf_global = Ridge(alpha=1, random_state=42, solver="auto")
clf_global.fit(X_all, y_all)

print("Global Ridge model trained on all studies.")

Global Ridge model trained on all studies.


## 6. Global Evaluation

In [30]:
y_pred_all = clf_global.predict(X_all)

global_metrics = compute_metrics(y_all, y_pred_all, label="GLOBAL (all studies)")
print_metrics(global_metrics)


  GLOBAL (all studies)  (n=4334)
  MAE (raw)              : 0.8861
  MAE (rounded ×0.5)     : 0.8829
  MAE negative (y≤4)     : 1.0286  [n=1219]
  MAE rounded  negative  : 1.0172
  MAE positive (y>4)     : 0.8304  [n=3115]
  MAE rounded  positive  : 0.8303


## 7. Per-Study Evaluation

In [31]:
per_study_metrics = {}

for label, info in study_data.items():
    mask = study_ids == label
    X_s  = X_all[mask]
    y_s  = y_all[mask]

    y_pred_s = clf_global.predict(X_s)

    m = compute_metrics(y_s, y_pred_s, label=f"Global model → {label}")
    per_study_metrics[label] = m
    print_metrics(m)


  Global model → Study 1  (n=2602)
  MAE (raw)              : 0.8493
  MAE (rounded ×0.5)     : 0.8459
  MAE negative (y≤4)     : 1.0451  [n=628]
  MAE rounded  negative  : 1.0358
  MAE positive (y>4)     : 0.7870  [n=1974]
  MAE rounded  positive  : 0.7855

  Global model → Study 3  (n=392)
  MAE (raw)              : 1.3761
  MAE (rounded ×0.5)     : 1.3750
  MAE negative (y≤4)     : 1.3977  [n=187]
  MAE rounded  negative  : 1.3930
  MAE positive (y>4)     : 1.3564  [n=205]
  MAE rounded  positive  : 1.3585

  Global model → Study 4  (n=623)
  MAE (raw)              : 0.8559
  MAE (rounded ×0.5)     : 0.8483
  MAE negative (y≤4)     : 0.8549  [n=248]
  MAE rounded  negative  : 0.8266
  MAE positive (y>4)     : 0.8565  [n=375]
  MAE rounded  positive  : 0.8627

  Global model → Study 5  (n=717)
  MAE (raw)              : 0.7783
  MAE (rounded ×0.5)     : 0.7782
  MAE negative (y≤4)     : 0.7957  [n=156]
  MAE rounded  negative  : 0.7949
  MAE positive (y>4)     : 0.7735  [n=561]
  MA

## 8. Summary Table

In [32]:
rows = [global_metrics] + list(per_study_metrics.values())

summary = pd.DataFrame(rows)[[
    "label", "n_total",
    "mae", "mae_rounded",
    "mae_neg", "mae_rounded_neg",
    "mae_pos", "mae_rounded_pos",
]].rename(columns={
    "label":            "Evaluation Scope",
    "n_total":          "N",
    "mae":              "MAE",
    "mae_rounded":      "MAE (rounded)",
    "mae_neg":          "MAE neg (y≤4)",
    "mae_rounded_neg":  "MAE neg rounded",
    "mae_pos":          "MAE pos (y>4)",
    "mae_rounded_pos":  "MAE pos rounded",
})

summary = summary.set_index("Evaluation Scope")
summary = summary.round(4)
summary

,N,MAE,MAE (rounded),MAE neg (y≤4),MAE neg rounded,MAE pos (y>4),MAE pos rounded
Evaluation Scope,,,,,,,
GLOBAL (all studies),4334,0.8861,0.8829,1.0286,1.0172,0.8304,0.8303
Global model → Study 1,2602,0.8493,0.8459,1.0451,1.0358,0.7870,0.7855
Global model → Study 3,392,1.3761,1.3750,1.3977,1.3930,1.3564,1.3585
Global model → Study 4,623,0.8559,0.8483,0.8549,0.8266,0.8565,0.8627
Global model → Study 5,717,0.7783,0.7782,0.7957,0.7949,0.7735,0.7736


## 9. Comparison: Global Model vs. Per-Study Models

Re-train individual study models and compare their in-sample MAE
against the global model evaluated on the same data.

In [33]:
comparison_rows = []

for label, info in study_data.items():
    X_s = info["X"]
    y_s = info["y"]

    # Individual study model
    clf_s = Ridge(alpha=1, random_state=42, solver="auto")
    clf_s.fit(X_s, y_s)
    y_pred_indiv   = clf_s.predict(X_s)
    y_pred_rounded = np.round(y_pred_indiv * 2) / 2
    mae_indiv      = mean_absolute_error(y_s, y_pred_indiv)
    mae_indiv_r    = mean_absolute_error(y_s, y_pred_rounded)

    # Global model on same data
    mae_global   = per_study_metrics[label]["mae"]
    mae_global_r = per_study_metrics[label]["mae_rounded"]

    comparison_rows.append({
        "Study":                  label,
        "N":                      len(y_s),
        "MAE (study model)":      round(mae_indiv, 4),
        "MAE (global model)":     round(mae_global, 4),
        "Δ MAE (global–study)":   round(mae_global - mae_indiv, 4),
        "MAE r (study model)":    round(mae_indiv_r, 4),
        "MAE r (global model)":   round(mae_global_r, 4),
        "Δ MAE r (global–study)": round(mae_global_r - mae_indiv_r, 4),
    })

comp_df = pd.DataFrame(comparison_rows).set_index("Study")
print("\n=== Global model vs. individual study models ===")
print("Positive Δ = global model is worse; Negative Δ = global model is better")
comp_df


=== Global model vs. individual study models ===
Positive Δ = global model is worse; Negative Δ = global model is better


,N,MAE (study model),MAE (global model),Δ MAE (global–study),MAE r (study model),MAE r (global model),Δ MAE r (global–study)
Study,,,,,,,
Study 1,2602,0.8033,0.8493,0.0460,0.7923,0.8459,0.0536
Study 3,392,1.0476,1.3761,0.3284,1.0446,1.3750,0.3304
Study 4,623,0.6390,0.8559,0.2168,0.6364,0.8483,0.2119
Study 5,717,0.5901,0.7783,0.1882,0.5642,0.7782,0.2141


## 10. Save the Global Model

In [34]:
import os

save_dir = '../../final_pipeline/models/satisfaction_final/'
os.makedirs(save_dir, exist_ok=True)

with open(os.path.join(save_dir, 'ridge_model_global.pkl'), 'wb') as f:
    pickle.dump(clf_global, f)

print("Global model saved to:", os.path.join(save_dir, 'ridge_model_global.pkl'))

Global model saved to: ../../final_pipeline/models/satisfaction_final/ridge_model_global.pkl


## 11. Add Predictions to Each Study DataFrame & Export

In [ ]:
# for label, info in study_data.items():
#     df         = info["df"].copy()
#     sat_col    = info["sat_col"]
#     y_pred_s   = clf_global.predict(info["X"])
#     y_pred_r   = np.round(y_pred_s * 2) / 2

#     pred_col = f"pred_{sat_col}_global"
#     df[pred_col] = y_pred_r

#     # Reorder columns: put prediction right after ground truth
#     cols = list(df.columns)
#     sat_idx = cols.index(sat_col)
#     cols.remove(pred_col)
#     cols.insert(sat_idx + 1, pred_col)
#     df = df[cols]

#     study_data[label]["df_with_preds"] = df
#     print(f"{label}: predictions added as '{pred_col}'")
#     display(df.head(3))